# 06 · Reserves and Basel III-style stress tests

Daily balance sheets for Tether and Circle are reconstructed from quarterly attestations with duration-based
mark-to-market (T-bills on the 3M Treasury yield, repo on SOFR, corporate bonds on AA curves, BTC and gold at
market). Equity cushion = reserves − tokens outstanding.

| Scenario | Shock |
|---|---|
| A · Liquidity run | 30% of supply redeemed; shortfall over cash + overnight repo sold at a 2% fire-sale haircut |
| B · Flash crash | BTC −50%, gold −15%, +300 bp rate shock through T-bill duration |
| C · Credit contagion | B + 10% corporate-bond default, 25% secured-loan and 50% other-investment haircuts |

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import reserves, stress_testing as st, plotting
from stablecoin_risk.data import loaders

q = reserves.quarterly_equity()
fig = plotting.plot_reserve_evolution(q)

In [3]:
fig = plotting.plot_reserve_composition(reserves.tether_composition(), reserves.circle_composition())

Tether runs a *shadow-bank* balance sheet (≈20% in corporate bonds, secured loans, BTC and gold) with a 3–6% equity
buffer; Circle runs a *narrow bank* (cash, T-bills, repo) with an equity buffer that rounds to zero.

In [4]:
out = {}
for issuer in ("tether", "circle"):
    bs = loaders.load_reserves(issuer)
    cushions = st.run_stress_tests(bs, issuer)
    out[issuer] = st.summarize(cushions, bs[st.ISSUERS[issuer].supply])
    fig = plotting.plot_stress_test(cushions, f"{issuer.title()}: equity cushion under stress")
pd.concat(out, names=["issuer", "scenario"])

days_insolvent_pct median_cushion_pct_supply min_cushion_bn min_cushion_date svb_window_min_bn svb_window_days_insolvent  \
issuer scenario                                                                                                                                       
tether Baseline                        0.8071                     3.989        -0.6019       2022-10-12           -0.3016                         1   
       A: Liquidity run                 3.067                       3.6        -0.8365       2022-10-12           -0.6011                         1   
       B: Flash crash                   35.11                       1.2         -3.289       2026-01-30            -2.381                         9   
       C: Credit contagion              89.67                    -1.679         -8.932       2026-01-30            -4.398                        12   
circle Baseline                         11.22                    0.1673         -1.411       2022-10-12            -1.223                         1   
       A: Liquidity run                 14.93                    0.1628         -1.503       2022-10-12            -1.228                         1   
       B: Flash crash                   33.33                   0.04921         -1.541       2022-10-12            -1.341                         5   
       C: Credit contagion              33.33                   0.04921         -1.541       2022-10-12            -1.341                         5   

                           svb_window_verdict  
issuer scenario                                
tether Baseline                          FAIL  
       A: Liquidity run                  FAIL  
       B: Flash crash                    FAIL  
       C: Credit contagion               FAIL  
circle Baseline                          FAIL  
       A: Liquidity run                  FAIL  
       B: Flash crash                    FAIL  
       C: Credit contagion               FAIL

In [5]:
pd.DataFrame({i: {"liquidity coverage of a 30% run": st.liquidity_coverage(loaders.load_reserves(i), i).loc["2023-03-10"],
                  "equity / reserves": st.equity_ratio(loaders.load_reserves(i), i).loc["2023-03-10"]}
              for i in ("tether", "circle")}).T.round(3).rename_axis("as of 2023-03-10")

,liquidity coverage of a 30% run,equity / reserves
as of 2023-03-10,,
tether,0.374,0.022
circle,0.813,0.016


**The central result.** Under the run scenario that actually happened in March 2023, Circle's cushion is negative
on **~15% of days** versus **~3%** for Tether. Circle held *more* liquid assets relative to a 30% run (coverage 0.81
vs 0.37), but its equity buffer is too thin to absorb even a small fire-sale loss. Tether's riskier book fails the
market and credit scenarios (B, C), which shows what the GENIUS Act's asset rules are designed to prevent, but its
equity buffer is what carried it through a run.

**Caveat.** The daily reconstruction interpolates quarterly attestations against daily market caps, so the cushion
is noisy day to day. The headline metric is therefore the *share of days insolvent*, not any single date.